# H22b: Extreme Anisotropy -- Effective-Rank Tail Proxy Study

## Counterpart identity

This notebook is the explanatory counterpart to:
`experiments/Tier_4_Falsification_Stress_Tests/H22b_EXTREME_ANISOTROPY_NOISE_EQUALIZATION/run_experiment.py`

Unlike the previous notebook version, this notebook **does not re-implement the full experiment core**.
It imports the script module, runs its structured `run_experiment()` entrypoint, and then uses the
returned raw results for tables, figures, and interpretation.

## Truthful scope

This pair currently studies a **deterministic full-batch single-layer linear-regression toy problem**.
It varies the synthetic input anisotropy through a data-side condition parameter $\kappa$ and tracks:

- the initial gradient's effective rank,
- an **effective-rank tail proxy** for how much of Muon's equalized update would be allocated to the
  low-energy singular-value complement,
- Muon's and fixed-$k$ Muon-clip's final-loss advantage over SGD.

This notebook is therefore a serious presentation of a **low-energy-tail proxy study**, not a direct
measurement of stochastic gradient-noise singular vectors.

## Why this notebook exists

The goal of this first completion pass is not to redesign the scientific question. Instead it makes
this H22b pair more honest and reusable:

1. the script is import-safe and returns structured results,
2. the notebook uses those returned results instead of duplicating the experiment core,
3. the narrative now matches the implemented metrics,
4. the conclusion is calibrated to the observed T1/T2 outcomes.

In [ ]:
from pathlib import Path
import importlib.util
import math
import os
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    def display(obj):
        print(obj)

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3


def display_markdown(text):
    if Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def display_table(rows, columns=None, precision=4):
    if pd is not None:
        df = pd.DataFrame(rows)
        if columns is not None:
            df = df[columns]
        display(df)
        return df

    if not rows:
        print('(empty table)')
        return rows

    if columns is None:
        columns = list(rows[0].keys())

    formatted_rows = []
    for row in rows:
        formatted = {}
        for col in columns:
            value = row[col]
            if isinstance(value, float):
                if math.isfinite(value):
                    formatted[col] = f'{value:.{precision}f}'
                else:
                    formatted[col] = str(value)
            else:
                formatted[col] = str(value)
        formatted_rows.append(formatted)

    widths = {col: max(len(col), max(len(r[col]) for r in formatted_rows)) for col in columns}
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    divider = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in formatted_rows:
        print(' | '.join(row[col].ljust(widths[col]) for col in columns))
    return rows


In [ ]:
RELATIVE_EXPERIMENT_DIR = Path('experiments/Tier_4_Falsification_Stress_Tests/H22b_EXTREME_ANISOTROPY_NOISE_EQUALIZATION')


def resolve_experiment_dir():
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base, base / RELATIVE_EXPERIMENT_DIR])

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate not in seen:
            seen.add(candidate)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        script_path = candidate / 'run_experiment.py'
        notebook_path = candidate / 'run_experiment.ipynb'
        if script_path.exists() and notebook_path.exists():
            return candidate

    raise FileNotFoundError(
        'Could not locate the H22b experiment directory from the current working directory. '
        'Tried current directory, its parents, and the known project-relative experiment path.'
    )


EXPERIMENT_DIR = resolve_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'

spec = importlib.util.spec_from_file_location('h22b_extreme_anisotropy', SCRIPT_PATH)
h22b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h22b)

print('Resolved experiment directory:', EXPERIMENT_DIR)
print('Resolved script path       :', SCRIPT_PATH)
print('Resolved notebook path     :', NOTEBOOK_PATH)
print('Import-safe entrypoint     : h22b.run_experiment(...)')

## Reproducibility, runtime, and configuration

The script defaults are intentionally preserved from the original pair:

- `DIM = 32`, `BATCH_SIZE = 64`, `NUM_STEPS = 500`
- `NUM_SEEDS = 5`
- `KAPPA_VALUES = [1, 10, 100, 1000, 10000]`
- 12 learning rates each for SGD, Muon, and Muon-clip

That default run performs **615 training calls** (`540` during LR tuning and `75` during final
five-seed evaluation), so a full execution is expected to take **tens of seconds** on CPU.

For automated smoke execution, this notebook also supports an environment variable:
`H22B_NOTEBOOK_SMOKE=1`
which switches to a tiny reduced configuration without changing the default scientific setup.

In [ ]:
SMOKE_MODE = os.environ.get('H22B_NOTEBOOK_SMOKE', '0') == '1'

if SMOKE_MODE:
    runtime_overrides = {
        'num_steps': 20,
        'num_seeds': 2,
        'diagnostic_seed_count': 2,
        'lr_search_seed_count': 2,
        'kappa_values': [1, 100],
        'lr_sgd': h22b.LR_SGD[:2],
        'lr_muon': h22b.LR_MUON[:2],
        'lr_clip': h22b.LR_CLIP[:2],
    }
else:
    runtime_overrides = {}

full_total_calls = len(h22b.KAPPA_VALUES) * (len(h22b.LR_SGD) + len(h22b.LR_MUON) + len(h22b.LR_CLIP)) * 3 + len(h22b.KAPPA_VALUES) * 3 * h22b.NUM_SEEDS

print('Timestamp              :', datetime.now().isoformat(timespec='seconds'))
print('Working directory      :', Path.cwd())
print('Smoke mode             :', SMOKE_MODE)
print('Default total train calls:', full_total_calls)
print('Default seeds          :', [h22b.BASE_SEED + i * h22b.SEED_STRIDE for i in range(h22b.NUM_SEEDS)])
print('Runtime overrides      :', runtime_overrides if runtime_overrides else '(none; full default run)')

## Caveat: what the proxy does and does not measure

The core diagnostic in this pair is a **tail-fraction proxy derived from rounded entropy effective rank**.
For an initial gradient $G$, the script:

1. computes the entropy effective rank of $G$,
2. rounds it to an integer $k$,
3. labels the top-$k$ singular directions as the retained signal proxy,
4. labels the remaining $d-k$ directions as the equalized tail proxy.

So the reported quantity is:

$$\text{tail\_fraction\_proxy} = 1 - k/d$$

with an accompanying gradient-energy concentration diagnostic.

For the Muon-clip comparison in this pair, that rank is used in a **fixed-$k$ per-$\kappa$** way: the script
chooses one clipping rank from the mean rounded initial effective rank across the diagnostic seeds, then keeps
that $k$ fixed during the subsequent training runs. It is therefore **not** a truly adaptive per-step rule here.

This is **not** a direct decomposition of stochastic signal and noise in $\mathrm{ortho}(G)$, because this
pair does not currently inject or measure stochastic gradient noise. The language in this notebook has been
narrowed accordingly.

## Execute the shared experiment core

The next cell calls the script's `run_experiment()` function directly. This is the single source of truth
for all metrics shown below.

In [ ]:
run_start = time.time()
results = h22b.run_experiment(verbose=True, **runtime_overrides)
notebook_elapsed = time.time() - run_start

summary_rows = results['summary_rows']
kappa_results = results['kappa_results']
correlations = results['correlations']
tests = results['tests']
config = results['config']

print('\nNotebook wall-clock time: %.2fs' % notebook_elapsed)
print('Script-reported runtime : %.2fs' % results['runtime_seconds'])
print('Observed train calls    :', config['total_train_calls'])

## High-level summary tables

The first table aggregates each $\kappa$ into the diagnostics and final losses reported by the script.
The second table exposes the per-optimizer best learning rates and finite-run counts so that the reported
means are easier to interpret.

In [ ]:
summary_table = []
best_lr_table = []
for entry in kappa_results:
    muon_clip_key = entry['muon_clip_key']
    sgd_eval = entry['final_eval']['sgd']
    muon_eval = entry['final_eval']['muon']
    clip_eval = entry['final_eval'][muon_clip_key]
    summary_table.append({
        'kappa': int(entry['kappa']),
        'eff_rank_cont': entry['diagnostics']['means']['effective_rank_continuous'],
        'tail_proxy': entry['diagnostics']['means']['tail_fraction_proxy'],
        'grad_signal_proxy': entry['diagnostics']['means']['gradient_signal_proxy'],
        'k_clip': entry['k_clip'],
        'sgd_mean_loss': sgd_eval['mean_finite_loss'],
        'muon_mean_loss': muon_eval['mean_finite_loss'],
        'muon_clip_mean_loss': clip_eval['mean_finite_loss'],
        'muon_advantage': entry['advantages']['muon_advantage'],
        'muon_clip_advantage': entry['advantages']['muon_clip_advantage'],
    })
    best_lr_table.append({
        'kappa': int(entry['kappa']),
        'best_lr_sgd': entry['best_lrs']['sgd'],
        'best_lr_muon': entry['best_lrs']['muon'],
        'best_lr_muon_clip': entry['best_lrs'][muon_clip_key],
        'sgd_finite': f"{sgd_eval['finite_count']}/{sgd_eval['total_count']}",
        'muon_finite': f"{muon_eval['finite_count']}/{muon_eval['total_count']}",
        'muon_clip_finite': f"{clip_eval['finite_count']}/{clip_eval['total_count']}",
    })

print('Aggregate summary by kappa')
display_table(
    summary_table,
    columns=[
        'kappa', 'eff_rank_cont', 'tail_proxy', 'grad_signal_proxy', 'k_clip',
        'sgd_mean_loss', 'muon_mean_loss', 'muon_clip_mean_loss',
        'muon_advantage', 'muon_clip_advantage'
    ],
    precision=4,
)

print('\nBest learning rates and final-eval finite counts')
display_table(
    best_lr_table,
    columns=[
        'kappa', 'best_lr_sgd', 'best_lr_muon', 'best_lr_muon_clip',
        'sgd_finite', 'muon_finite', 'muon_clip_finite'
    ],
    precision=6,
)

## Diagnostic view: effective rank and tail proxy vs $\kappa$

This figure shows the initial-gradient diagnostics returned by the script. Seed-wise points are plotted so
that the reader can see how much variability exists before aggregation. The first panel shows the entropy
continuous effective rank; the second shows the rounded-rank tail proxy and the gradient-energy concentration
captured by the retained top-$k$ singular directions.

In [ ]:
diagnostic_rows = []
for entry in kappa_results:
    for record in entry['diagnostics']['records']:
        diagnostic_rows.append({
            'kappa': int(entry['kappa']),
            'seed': record['seed'],
            'effective_rank_continuous': record['effective_rank_continuous'],
            'effective_rank_rounded': record['effective_rank_rounded'],
            'tail_fraction_proxy': record['tail_fraction_proxy'],
            'gradient_signal_proxy': record['gradient_signal_proxy'],
            'grad_condition_proxy': record['grad_condition_proxy'],
        })

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

kappa_axis = np.array([row['kappa'] for row in summary_table], dtype=float)

for seed in sorted({row['seed'] for row in diagnostic_rows}):
    seed_rows = [row for row in diagnostic_rows if row['seed'] == seed]
    axes[0].plot(
        [row['kappa'] for row in seed_rows],
        [row['effective_rank_continuous'] for row in seed_rows],
        marker='o', alpha=0.45, linewidth=1,
    )
    axes[1].plot(
        [row['kappa'] for row in seed_rows],
        [row['tail_fraction_proxy'] for row in seed_rows],
        marker='o', alpha=0.35, linewidth=1,
    )

axes[0].plot(kappa_axis, [row['eff_rank_cont'] for row in summary_table], color='black', marker='o', linewidth=2.5, label='mean effective rank')
axes[0].set_xscale('log')
axes[0].set_xlabel('kappa')
axes[0].set_ylabel('effective rank (continuous)')
axes[0].set_title('Initial-gradient effective rank')
axes[0].legend(loc='best')

axes[1].plot(kappa_axis, [row['tail_proxy'] for row in summary_table], color='tab:red', marker='o', linewidth=2.5, label='mean tail proxy')
axes[1].plot(kappa_axis, [row['grad_signal_proxy'] for row in summary_table], color='tab:blue', marker='s', linewidth=2.0, label='mean gradient signal proxy')
axes[1].set_xscale('log')
axes[1].set_xlabel('kappa')
axes[1].set_ylabel('fraction')
axes[1].set_title('Equalized tail proxy and retained energy')
axes[1].set_ylim(-0.02, 1.02)
axes[1].legend(loc='best')

plt.tight_layout()
plt.show()

## Performance view: Muon and fixed-$k$ Muon-clip advantage vs $\kappa$

The left panel tracks the optimizer advantages relative to SGD using the script's aggregate mean-finite-loss
metric. The right panel shows the corresponding final mean losses themselves. The Muon-clip curve here is the
implemented **fixed-$k$ per-$\kappa$** variant, where $k$ is chosen once from the initial diagnostic seeds and
not updated during training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(kappa_axis, [row['muon_advantage'] for row in summary_table], marker='o', linewidth=2.5, label='Muon / SGD advantage')
axes[0].plot(kappa_axis, [row['muon_clip_advantage'] for row in summary_table], marker='s', linewidth=2.5, label='Muon-clip / SGD advantage')
for row in summary_table:
    axes[0].annotate(f"k={row['k_clip']}", (row['kappa'], row['muon_clip_advantage']), textcoords='offset points', xytext=(4, 4), fontsize=8)
axes[0].set_xscale('log')
axes[0].set_xlabel('kappa')
axes[0].set_ylabel('advantage = mean loss(SGD) / mean loss(optimizer)')
axes[0].set_title('Optimizer advantage vs anisotropy')
axes[0].legend(loc='best')

axes[1].plot(kappa_axis, [row['sgd_mean_loss'] for row in summary_table], marker='o', linewidth=2.0, label='SGD')
axes[1].plot(kappa_axis, [row['muon_mean_loss'] for row in summary_table], marker='o', linewidth=2.0, label='Muon')
axes[1].plot(kappa_axis, [row['muon_clip_mean_loss'] for row in summary_table], marker='o', linewidth=2.0, label='Muon-clip')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('kappa')
axes[1].set_ylabel('final mean finite loss')
axes[1].set_title('Final mean losses')
axes[1].legend(loc='best')

plt.tight_layout()
plt.show()

## Correlation view: tail proxy vs Muon advantage

The script's T1 statistic correlates the aggregated tail-fraction proxy with the log of Muon's advantage.
Because there are only five default $\kappa$ points, this should be read as a descriptive summary rather than
strong statistical evidence.

In [ ]:
tail_proxy_corr = correlations['tail_fraction_proxy_vs_log_muon_advantage']

fig, ax = plt.subplots(figsize=(6.5, 5))
xs = np.array([row['tail_proxy'] for row in summary_table], dtype=float)
ys = np.log(np.clip(np.array([row['muon_advantage'] for row in summary_table], dtype=float), 1e-10, None))
ax.scatter(xs, ys, s=80)
for row, x, y in zip(summary_table, xs, ys):
    ax.annotate(f"κ={row['kappa']}", (x, y), textcoords='offset points', xytext=(6, 6))

if len(xs) >= 2 and np.std(xs) > 0:
    coeffs = np.polyfit(xs, ys, deg=1)
    line_x = np.linspace(xs.min(), xs.max(), 100)
    line_y = coeffs[0] * line_x + coeffs[1]
    ax.plot(line_x, line_y, linestyle='--', color='black', alpha=0.7)

ax.set_xlabel('tail-fraction proxy')
ax.set_ylabel('log(Muon advantage)')
ax.set_title(f"Correlation view: r = {tail_proxy_corr:.3f}")
plt.tight_layout()
plt.show()

print('T1 observed correlation:', tail_proxy_corr)
print('T1 pass status         :', tests['T1']['pass'])

## Per-seed final losses and stability diagnostics

The script keeps the original mean-of-finite-losses reporting rule, but it now also exposes per-seed final
losses and finite counts. This makes it easier to see whether an apparent mean is driven by a subset of
successful seeds.

In [ ]:
loss_rows = []
for entry in kappa_results:
    muon_clip_key = entry['muon_clip_key']
    sgd_losses = entry['final_eval']['sgd']['losses']
    muon_losses = entry['final_eval']['muon']['losses']
    clip_losses = entry['final_eval'][muon_clip_key]['losses']
    for seed, sgd_loss, muon_loss, clip_loss in zip(config['seeds'], sgd_losses, muon_losses, clip_losses):
        loss_rows.append({
            'kappa': int(entry['kappa']),
            'seed': seed,
            'sgd_loss': sgd_loss,
            'muon_loss': muon_loss,
            'muon_clip_loss': clip_loss,
            'muon_advantage_pairwise': sgd_loss / max(muon_loss, 1e-30) if np.isfinite(sgd_loss) and np.isfinite(muon_loss) else np.inf,
            'muon_clip_advantage_pairwise': sgd_loss / max(clip_loss, 1e-30) if np.isfinite(sgd_loss) and np.isfinite(clip_loss) else np.inf,
        })

print('Per-seed final-loss records')
display_table(
    loss_rows,
    columns=[
        'kappa', 'seed', 'sgd_loss', 'muon_loss', 'muon_clip_loss',
        'muon_advantage_pairwise', 'muon_clip_advantage_pairwise'
    ],
    precision=6,
)

## Calibrated interpretation and conclusion

This final cell translates the returned metrics into a conclusion that is faithful to the implemented toy
study. In particular, it explicitly references the T1/T2 outcomes instead of narrating the originally hoped-for
mechanism as though it had already been established.

In [ ]:
t1 = tests['T1']
t2 = tests['T2']
max_tail_row = max(summary_table, key=lambda row: row['tail_proxy'])
max_muon_row = max(summary_table, key=lambda row: row['muon_advantage'])
conclusion_lines = [
    '### Observed outcome summary',
    '',
    f"- The proxy tail fraction increases across the anisotropy sweep and reaches its maximum at κ={max_tail_row['kappa']}.",
    f"- Muon's largest observed advantage occurs at κ={max_muon_row['kappa']}, not at the isotropic endpoint.",
    f"- **T1** (`tail proxy` vs `log Muon advantage`) is **{'PASS' if t1['pass'] else 'FAIL'}** with observed correlation $r = {t1['observed_r']:.3f}$.",
    f"- **T2** (fixed-$k$ Muon-clip beats Muon by >20% at the highest κ) is **{'PASS' if t2['pass'] else 'FAIL'}**.",
    '',
    '### What this notebook supports',
    '',
    '- It supports the claim that the synthetic setup produces progressively lower effective rank and a larger effective-rank tail proxy as κ increases.',
    '- It supports a careful optimizer comparison for this deterministic full-batch toy problem.',
    '',
    '### What this notebook does not support',
    '',
    '- It does **not** directly demonstrate stochastic noise singular vectors being amplified by Muon, because no stochastic noise decomposition is measured here.',
    '- It does **not** currently show the predicted negative correlation between the proxy and Muon advantage when T1 fails.',
    '- It does **not** currently show the fixed-$k$ Muon-clip rescue story when T2 fails.',
    '',
    '### Calibrated conclusion',
    '',
    'This first-pass pair is now aligned and academically honest: the notebook matches the script, uses its returned raw results, and presents the current implementation as an **effective-rank / low-energy-tail proxy stress test**. The present implementation should therefore be read as a falsification-oriented toy study rather than a completed mechanistic confirmation of literal noise equalization.'
]

if SMOKE_MODE:
    conclusion_lines.extend([
        '',
        '> **Smoke-mode note:** this execution used `H22B_NOTEBOOK_SMOKE=1`, so the numerical values above are only a usability check. Re-run without that environment variable for the full default study.'
    ])

display_markdown('\n'.join(conclusion_lines))